# Wav2Vec2 Fine-Tuning Baseline

This notebook reuses the processed participant audio in `processed/isolated_audio`, creates segment-level training examples, fine-tunes a pretrained `facebook/wav2vec2-base-960h` model for PHQ score regression, and evaluates predictions back at the participant level.

Compared with the frozen-encoder baseline, this version:
- keeps the same participant audio inputs
- trains on overlapping audio segments
- freezes the low-level frontend and lower encoder blocks
- unfreezes the feature projection, the last few transformer layers, and the regression head


In [33]:
from __future__ import annotations

import json
import math
import random
from collections import defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
import soundfile as sf
import torch
from scipy.signal import resample_poly
from sklearn.metrics import mean_absolute_error, mean_squared_error
from torch.optim import AdamW
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm
from transformers import (
    AutoConfig,
    AutoProcessor,
    Wav2Vec2ForSequenceClassification,
    get_linear_schedule_with_warmup,
)

BASE_DIR = Path('.')
ISOLATED_AUDIO_DIR = BASE_DIR / 'processed' / 'isolated_audio'
METADATA_XLSX = BASE_DIR / 'processed' / 'spectrograms' / 'segment_metadata.xlsx'
OUTPUT_ROOT = BASE_DIR / 'processed' / 'wav2vec_finetune'
MODEL_ROOT = BASE_DIR / 'best_model' / 'wav2vec_finetune'

HF_MODEL_NAME = 'facebook/wav2vec2-base-960h'
TARGET_SR = 16_000
SEGMENT_SEC = 8.0
HOP_SEC = 4.0
TRAIN_BATCH_SIZE = 4
EVAL_BATCH_SIZE = 8
GRAD_ACCUM_STEPS = 2
NUM_EPOCHS = 5
LEARNING_RATE = 5e-6
WEIGHT_DECAY = 1e-4
MAX_GRAD_NORM = 1.0
WARMUP_RATIO = 0.1
UNFREEZE_LAST_N_LAYERS = 4
EARLY_STOPPING_PATIENCE = 2
NUM_WORKERS = 0
RANDOM_STATE = 42
SPLITS = ('train', 'dev', 'test')

DEVICE = torch.device(
    'mps' if torch.backends.mps.is_available()
    else 'cuda' if torch.cuda.is_available()
    else 'cpu'
)

RUN_NAME = (
    f'wav2vec2_ft_seg{int(SEGMENT_SEC)}_hop{int(HOP_SEC)}'
    f'_u{UNFREEZE_LAST_N_LAYERS}_lr{LEARNING_RATE:g}'
)
RUN_DIR = OUTPUT_ROOT / RUN_NAME
CACHE_NAME_lr_1e05 = 'wav2vec2_ft_seg8_hop4_u4_lr1e-05'
# SEGMENT_CACHE_DIR = RUN_DIR / 'segment_cache' (usually use this but to reuse the cache from lr 1e-05)
SEGMENT_CACHE_DIR = OUTPUT_ROOT / CACHE_NAME_lr_1e05 / 'segment_cache'


OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
MODEL_ROOT.mkdir(parents=True, exist_ok=True)
RUN_DIR.mkdir(parents=True, exist_ok=True)
SEGMENT_CACHE_DIR.mkdir(parents=True, exist_ok=True)

def set_seed(seed: int = RANDOM_STATE) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(RANDOM_STATE)

print(f'Audio dir      : {ISOLATED_AUDIO_DIR.resolve()}')
print(f'Metadata file  : {METADATA_XLSX.resolve()}')
print(f'Output root    : {OUTPUT_ROOT.resolve()}')
print(f'Run dir        : {RUN_DIR.resolve()}')
print(f'Model root     : {MODEL_ROOT.resolve()}')
print(f'Encoder        : {HF_MODEL_NAME}')
print(f'Device         : {DEVICE}')

if not ISOLATED_AUDIO_DIR.exists():
    raise FileNotFoundError(f'Isolated audio directory not found: {ISOLATED_AUDIO_DIR}')
if not METADATA_XLSX.exists():
    raise FileNotFoundError(f'Metadata spreadsheet not found: {METADATA_XLSX}')


Audio dir      : /Users/roshan/Documents/SUTD/Term 8/DL/Project/sutd_50.039_Deep_Learning/processed/isolated_audio
Metadata file  : /Users/roshan/Documents/SUTD/Term 8/DL/Project/sutd_50.039_Deep_Learning/processed/spectrograms/segment_metadata.xlsx
Output root    : /Users/roshan/Documents/SUTD/Term 8/DL/Project/sutd_50.039_Deep_Learning/processed/wav2vec_finetune
Run dir        : /Users/roshan/Documents/SUTD/Term 8/DL/Project/sutd_50.039_Deep_Learning/processed/wav2vec_finetune/wav2vec2_ft_seg8_hop4_u4_lr5e-06
Model root     : /Users/roshan/Documents/SUTD/Term 8/DL/Project/sutd_50.039_Deep_Learning/best_model/wav2vec_finetune
Encoder        : facebook/wav2vec2-base-960h
Device         : mps


In [34]:
metadata = pd.read_excel(METADATA_XLSX)
for col in ['participant_id', 'phq_score', 'binary_label', 'participant_dur_s', 'n_turns']:
    metadata[col] = pd.to_numeric(metadata[col], errors='coerce')

metadata['participant_id'] = metadata['participant_id'].astype(int)
metadata['split'] = metadata['split'].astype(str).str.strip().str.lower()

participant_df = (
    metadata.sort_values(['participant_id', 'segment_idx'])
    .groupby('participant_id', as_index=False)
    .agg({
        'split': 'first',
        'phq_score': 'first',
        'binary_label': 'first',
        'participant_dur_s': 'first',
        'n_turns': 'first',
    })
)

participant_df['session_name'] = participant_df['participant_id'].astype(str) + '_P'
participant_df['audio_path'] = participant_df['session_name'].map(lambda s: ISOLATED_AUDIO_DIR / f'{s}.wav')
participant_df['audio_exists'] = participant_df['audio_path'].map(Path.exists)

missing_audio = participant_df.loc[~participant_df['audio_exists'], ['participant_id', 'audio_path']]
if not missing_audio.empty:
    raise FileNotFoundError(
        'Missing isolated WAV files for some participants. Examples:\n' +
        missing_audio.head(10).to_string(index=False)
    )

display(participant_df.head())
print(participant_df['split'].value_counts().to_string())


,participant_id,split,phq_score,binary_label,participant_dur_s,n_turns,session_name,audio_path,audio_exists
0,300,test,2,0,155.76,87,300_P,processed/isolated_audio/300_P.wav,True
1,301,test,3,0,475.44,104,301_P,processed/isolated_audio/301_P.wav,True
2,302,dev,4,0,208.93,97,302_P,processed/isolated_audio/302_P.wav,True
3,303,train,0,0,642.93,103,303_P,processed/isolated_audio/303_P.wav,True
4,304,train,6,0,362.60,104,304_P,processed/isolated_audio/304_P.wav,True


split
train    107
test      47
dev       35


In [35]:
def load_audio_mono(path: Path, target_sr: int = TARGET_SR) -> np.ndarray:
    audio, sr = sf.read(path, always_2d=False)
    audio = np.asarray(audio, dtype=np.float32)
    if audio.ndim == 2:
        audio = audio.mean(axis=1)
    if sr != target_sr:
        audio = resample_poly(audio, target_sr, sr).astype(np.float32)
    return audio

def segment_audio(audio: np.ndarray, sample_rate: int, segment_sec: float, hop_sec: float) -> np.ndarray:
    segment_len = int(round(segment_sec * sample_rate))
    hop_len = int(round(hop_sec * sample_rate))
    if segment_len <= 0 or hop_len <= 0:
        raise ValueError('segment_sec and hop_sec must be positive')

    if len(audio) == 0:
        return np.zeros((1, segment_len), dtype=np.float32)

    segments = []
    start = 0
    while start + segment_len <= len(audio):
        segments.append(audio[start:start + segment_len])
        start += hop_len

    if not segments:
        padded = np.zeros(segment_len, dtype=np.float32)
        padded[:len(audio)] = audio
        segments.append(padded)

    return np.stack(segments).astype(np.float32)

sample_audio = load_audio_mono(participant_df.iloc[0]['audio_path'])
sample_segments = segment_audio(sample_audio, TARGET_SR, SEGMENT_SEC, HOP_SEC)
print('Sample audio length (sec):', len(sample_audio) / TARGET_SR)
print('Segment batch shape      :', sample_segments.shape)


Sample audio length (sec): 155.76
Segment batch shape      : (37, 128000)


In [36]:
segment_rows = []
segment_count_by_participant = {}

for row in tqdm(participant_df.itertuples(index=False), total=len(participant_df), desc='Preparing segment cache'):
    cache_file = SEGMENT_CACHE_DIR / f'{int(row.participant_id)}.npz'

    if cache_file.exists():
        cached = np.load(cache_file)
        n_segments = int(cached['segments'].shape[0])
    else:
        audio = load_audio_mono(row.audio_path)
        segments = segment_audio(audio, TARGET_SR, SEGMENT_SEC, HOP_SEC)
        n_segments = int(segments.shape[0])
        np.savez_compressed(
            cache_file,
            participant_id=int(row.participant_id),
            duration_s=float(len(audio) / TARGET_SR),
            segments=segments.astype(np.float32),
        )

    segment_count_by_participant[int(row.participant_id)] = n_segments

    for segment_idx in range(n_segments):
        segment_rows.append({
            'participant_id': int(row.participant_id),
            'session_name': str(row.session_name),
            'split': str(row.split),
            'phq_score': float(row.phq_score),
            'binary_label': int(row.binary_label),
            'participant_dur_s': float(row.participant_dur_s),
            'n_turns': int(row.n_turns),
            'cache_file': str(cache_file),
            'segment_idx': segment_idx,
            'n_segments': n_segments,
        })

segment_df = pd.DataFrame(segment_rows)
segment_df['participant_weight'] = 1.0 / segment_df['n_segments'].clip(lower=1)

display(segment_df.head())
print('Segments by split:')
print(segment_df['split'].value_counts().to_string())
print('Participants by split:')
print(participant_df['split'].value_counts().to_string())


Preparing segment cache: 100%|██████████| 189/189 [00:34<00:00,  5.49it/s]


,participant_id,session_name,split,phq_score,binary_label,participant_dur_s,n_turns,cache_file,segment_idx,n_segments,participant_weight
0,300,300_P,test,2.0,0,155.76,87,processed/wav2vec_finetune/wav2vec2_ft_seg8_ho...,0,37,0.027027
1,300,300_P,test,2.0,0,155.76,87,processed/wav2vec_finetune/wav2vec2_ft_seg8_ho...,1,37,0.027027
2,300,300_P,test,2.0,0,155.76,87,processed/wav2vec_finetune/wav2vec2_ft_seg8_ho...,2,37,0.027027
3,300,300_P,test,2.0,0,155.76,87,processed/wav2vec_finetune/wav2vec2_ft_seg8_ho...,3,37,0.027027
4,300,300_P,test,2.0,0,155.76,87,processed/wav2vec_finetune/wav2vec2_ft_seg8_ho...,4,37,0.027027


Segments by split:
split
train    11362
test      5904
dev       4322
Participants by split:
split
train    107
test      47
dev       35


In [17]:
processor = AutoProcessor.from_pretrained(HF_MODEL_NAME)

class SegmentDataset(Dataset):
    def __init__(self, frame: pd.DataFrame):
        self.frame = frame.reset_index(drop=True).copy()
        self._cache = {}

    def __len__(self) -> int:
        return len(self.frame)

    def _get_segment_bank(self, cache_file: str) -> np.ndarray:
        if cache_file not in self._cache:
            self._cache[cache_file] = np.load(cache_file)['segments'].astype(np.float32)
        return self._cache[cache_file]

    def __getitem__(self, idx: int) -> dict:
        row = self.frame.iloc[idx]
        segments = self._get_segment_bank(row.cache_file)
        audio = segments[int(row.segment_idx)]
        return {
            'audio': audio,
            'label': float(row.phq_score),
            'participant_id': int(row.participant_id),
            'binary_label': int(row.binary_label),
            'split': str(row.split),
            'segment_idx': int(row.segment_idx),
            'weight': float(row.participant_weight),
        }

class AudioRegressionCollator:
    def __init__(self, processor, sampling_rate: int = TARGET_SR):
        self.processor = processor
        self.sampling_rate = sampling_rate

    def __call__(self, batch: list[dict]) -> dict:
        inputs = self.processor(
            [item['audio'].tolist() for item in batch],
            sampling_rate=self.sampling_rate,
            return_tensors='pt',
            padding=True,
            return_attention_mask=True,
        )
        inputs['labels'] = torch.tensor([item['label'] for item in batch], dtype=torch.float32)
        inputs['weights'] = torch.tensor([item['weight'] for item in batch], dtype=torch.float32)
        inputs['participant_id'] = torch.tensor([item['participant_id'] for item in batch], dtype=torch.long)
        inputs['binary_label'] = torch.tensor([item['binary_label'] for item in batch], dtype=torch.long)
        inputs['segment_idx'] = torch.tensor([item['segment_idx'] for item in batch], dtype=torch.long)
        return inputs

split_frames = {split: segment_df.loc[segment_df['split'] == split].reset_index(drop=True) for split in SPLITS}
datasets = {split: SegmentDataset(frame) for split, frame in split_frames.items()}
collator = AudioRegressionCollator(processor)

train_loader = DataLoader(
    datasets['train'],
    batch_size=TRAIN_BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    collate_fn=collator,
)
eval_loaders = {
    split: DataLoader(
        datasets[split],
        batch_size=EVAL_BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        collate_fn=collator,
    )
    for split in SPLITS
}

for split in SPLITS:
    print(f'{split:5s} | participants={participant_df[participant_df.split == split].shape[0]:3d} | segments={split_frames[split].shape[0]:5d}')


train | participants=107 | segments=11362
dev   | participants= 35 | segments= 4322
test  | participants= 47 | segments= 5904


In [37]:
config = AutoConfig.from_pretrained(
    HF_MODEL_NAME,
    num_labels=1,
    problem_type='regression',
    final_dropout=0.1,
)

model = Wav2Vec2ForSequenceClassification.from_pretrained(
    HF_MODEL_NAME,
    config=config,
    ignore_mismatched_sizes=True,
)

model.config.apply_spec_augment = False
if hasattr(model.wav2vec2, "masked_spec_embed"):
    model.wav2vec2.masked_spec_embed = None
    

for param in model.parameters():
    param.requires_grad = False

for param in model.projector.parameters():
    param.requires_grad = True
for param in model.classifier.parameters():
    param.requires_grad = True
for param in model.wav2vec2.feature_projection.parameters():
    param.requires_grad = True
for param in model.wav2vec2.encoder.layer_norm.parameters():
    param.requires_grad = True
for layer in model.wav2vec2.encoder.layers[-UNFREEZE_LAST_N_LAYERS:]:
    for param in layer.parameters():
        param.requires_grad = True

model.to(DEVICE)

total_params = sum(param.numel() for param in model.parameters())
trainable_params = sum(param.numel() for param in model.parameters() if param.requires_grad)
print(f'Trainable params: {trainable_params:,} / {total_params:,} ({100 * trainable_params / total_params:.2f}%)')


Loading weights: 100%|██████████| 210/210 [00:00<00:00, 5535.92it/s]
Wav2Vec2ForSequenceClassification LOAD REPORT from: facebook/wav2vec2-base-960h
Key                        | Status     | 
---------------------------+------------+-
lm_head.weight             | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
projector.weight           | MISSING    | 
classifier.bias            | MISSING    | 
wav2vec2.masked_spec_embed | MISSING    | 
projector.bias             | MISSING    | 
classifier.weight          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Trainable params: 28,945,153 / 94,568,065 (30.61%)


In [39]:
def pearson_correlation(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    y_true = np.asarray(y_true, dtype=np.float32)
    y_pred = np.asarray(y_pred, dtype=np.float32)
    if y_true.size < 2:
        return float('nan')
    if np.isclose(y_true.std(), 0.0) or np.isclose(y_pred.std(), 0.0):
        return float('nan')
    return float(np.corrcoef(y_true, y_pred)[0, 1])

def regression_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> dict:
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    pearson_r = pearson_correlation(y_true, y_pred)
    return {
        'rmse': float(rmse),
        'mae': float(mae),
        'pearson_r': float(pearson_r),
    }

def move_batch_to_device(batch: dict, device: torch.device) -> dict:
    return {
        key: value.to(device) if isinstance(value, torch.Tensor) else value
        for key, value in batch.items()
    }

def collect_segment_predictions(model, loader: DataLoader) -> pd.DataFrame:
    model.eval()
    rows = []
    with torch.no_grad():
        for batch in tqdm(loader, total=len(loader), leave=False):
            batch = move_batch_to_device(batch, DEVICE)
            participant_id = batch.pop('participant_id').detach().cpu().numpy()
            binary_label = batch.pop('binary_label').detach().cpu().numpy()
            segment_idx = batch.pop('segment_idx').detach().cpu().numpy()
            weights = batch.pop('weights')
            outputs = model(**batch)
            preds = outputs.logits.squeeze(-1).detach().cpu().numpy()
            labels = batch['labels'].detach().cpu().numpy()
            weights_np = weights.detach().cpu().numpy()

            for i in range(len(preds)):
                rows.append({
                    'participant_id': int(participant_id[i]),
                    'segment_idx': int(segment_idx[i]),
                    'binary_label': int(binary_label[i]),
                    'phq_score': float(labels[i]),
                    'prediction': float(preds[i]),
                    'weight': float(weights_np[i]),
                })

    return pd.DataFrame(rows)

def aggregate_to_participants(segment_predictions: pd.DataFrame) -> pd.DataFrame:
    return (
        segment_predictions.sort_values(['participant_id', 'segment_idx'])
        .groupby('participant_id', as_index=False)
        .agg({
            'phq_score': 'first',
            'binary_label': 'first',
            'prediction': 'mean',
        })
        .rename(columns={'prediction': 'predicted_phq_score'})
    )

def evaluate_participant_level(model, loader: DataLoader) -> tuple[pd.DataFrame, dict]:
    segment_predictions = collect_segment_predictions(model, loader)
    participant_predictions = aggregate_to_participants(segment_predictions)
    metrics = regression_metrics(
        participant_predictions['phq_score'].to_numpy(dtype=np.float32),
        participant_predictions['predicted_phq_score'].to_numpy(dtype=np.float32),
    )
    return participant_predictions, metrics


In [25]:
# model.config.apply_spec_augment = False

# if hasattr(model.wav2vec2, "masked_spec_embed"):
#     model.wav2vec2.masked_spec_embed = None

# batch = next(iter(train_loader))
# batch = move_batch_to_device(batch, DEVICE)

# model.train()
# outputs = model(
#     input_values=batch['input_values'],
#     attention_mask=batch['attention_mask'],
# )
# preds = outputs.logits.squeeze(-1)
# labels = batch['labels']
# weights = batch['weights']
# loss = ((preds - labels) ** 2 * weights).mean()

# print('preds finite:', torch.isfinite(preds).all().item())
# print('loss finite:', torch.isfinite(loss).item())
# print(preds[:5])


preds finite: True
loss finite: True
tensor([-0.0061, -0.0002, -0.0042, -0.0075], grad_fn=<SliceBackward0>)


In [40]:
# batch = next(iter(train_loader))
# batch = move_batch_to_device(batch, DEVICE)

# model.train()
# outputs = model(
#     input_values=batch['input_values'],
#     attention_mask=batch['attention_mask'],
# )
# preds = outputs.logits.squeeze(-1)
# loss = ((preds - batch['labels']) ** 2 * batch['weights']).mean()

# print(torch.isfinite(preds).all().item(), torch.isfinite(loss).item())


True True


In [43]:
print("RUN_DIR:", RUN_DIR.resolve())
print("SEGMENT_CACHE_DIR:", SEGMENT_CACHE_DIR.resolve())


RUN_DIR: /Users/roshan/Documents/SUTD/Term 8/DL/Project/sutd_50.039_Deep_Learning/processed/wav2vec_finetune/wav2vec2_ft_seg8_hop4_u4_lr5e-06
SEGMENT_CACHE_DIR: /Users/roshan/Documents/SUTD/Term 8/DL/Project/sutd_50.039_Deep_Learning/processed/wav2vec_finetune/wav2vec2_ft_seg8_hop4_u4_lr1e-05/segment_cache


In [44]:
optimizer = AdamW(
    [param for param in model.parameters() if param.requires_grad],
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
)

steps_per_epoch = math.ceil(len(train_loader) / GRAD_ACCUM_STEPS)
total_training_steps = steps_per_epoch * NUM_EPOCHS
warmup_steps = int(total_training_steps * WARMUP_RATIO)
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_training_steps,
)

RESUME_FROM = None
SAVE_EVERY_N_STEPS = 200

start_epoch = 1
global_step = 0
best_dev_rmse = float('inf')
history = []
epochs_without_improvement = 0
best_checkpoint = RUN_DIR / 'best_model.pt'
latest_checkpoint = RUN_DIR / 'checkpoint_latest.pt'

if RESUME_FROM is not None and Path(RESUME_FROM).exists():
    checkpoint = torch.load(RESUME_FROM, map_location=DEVICE)
    model.load_state_dict(checkpoint['model_state_dict'])
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
    history = checkpoint.get('history', [])
    best_dev_rmse = checkpoint.get('best_dev_rmse', float('inf'))
    epochs_without_improvement = checkpoint.get('epochs_without_improvement', 0)
    global_step = checkpoint.get('global_step', 0)
    start_epoch = checkpoint['epoch'] + 1
    print(f"Resuming from epoch {checkpoint['epoch']}")

for epoch in range(start_epoch, NUM_EPOCHS + 1):
    model.train()
    optimizer.zero_grad(set_to_none=True)
    running_loss = 0.0

    progress = tqdm(train_loader, total=len(train_loader), desc=f'Epoch {epoch}/{NUM_EPOCHS}')
    for step, batch in enumerate(progress, start=1):
        batch = move_batch_to_device(batch, DEVICE)
        batch.pop('participant_id')
        batch.pop('binary_label')
        batch.pop('segment_idx')
        weights = batch.pop('weights')

        outputs = model(
            input_values=batch['input_values'],
            attention_mask=batch['attention_mask'],
        )
        preds = outputs.logits.squeeze(-1)
        labels = batch['labels']

        loss = ((preds - labels) ** 2 * weights).mean()

        if not torch.isfinite(loss):
            raise ValueError(f'Non-finite loss encountered at epoch={epoch}, step={step}: {loss.item()}')

        running_loss += float(loss.item())
        (loss / GRAD_ACCUM_STEPS).backward()

        if step % GRAD_ACCUM_STEPS == 0 or step == len(train_loader):
            torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad(set_to_none=True)
            global_step += 1

            if global_step % SAVE_EVERY_N_STEPS == 0:
                torch.save({
                    'epoch': epoch,
                    'global_step': global_step,
                    'model_state_dict': model.state_dict(),
                    'optimizer_state_dict': optimizer.state_dict(),
                    'scheduler_state_dict': scheduler.state_dict(),
                    'history': history,
                    'best_dev_rmse': best_dev_rmse,
                    'epochs_without_improvement': epochs_without_improvement,
                }, latest_checkpoint)

        progress.set_postfix(loss=running_loss / step)

    train_predictions, train_metrics = evaluate_participant_level(model, eval_loaders['train'])
    dev_predictions, dev_metrics = evaluate_participant_level(model, eval_loaders['dev'])

    train_predictions = train_predictions.replace([np.inf, -np.inf], np.nan).dropna(
        subset=['phq_score', 'predicted_phq_score']
    )
    dev_predictions = dev_predictions.replace([np.inf, -np.inf], np.nan).dropna(
        subset=['phq_score', 'predicted_phq_score']
    )

    train_metrics = regression_metrics(
        train_predictions['phq_score'].to_numpy(dtype=np.float32),
        train_predictions['predicted_phq_score'].to_numpy(dtype=np.float32),
    )
    dev_metrics = regression_metrics(
        dev_predictions['phq_score'].to_numpy(dtype=np.float32),
        dev_predictions['predicted_phq_score'].to_numpy(dtype=np.float32),
    )

    epoch_row = {
        'epoch': epoch,
        'train_loss': running_loss / max(len(train_loader), 1),
        'train_rmse': train_metrics['rmse'],
        'train_mae': train_metrics['mae'],
        'train_pearson_r': train_metrics['pearson_r'],
        'dev_rmse': dev_metrics['rmse'],
        'dev_mae': dev_metrics['mae'],
        'dev_pearson_r': dev_metrics['pearson_r'],
    }
    history.append(epoch_row)

    state = {
        'epoch': epoch,
        'global_step': global_step,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict(),
        'history': history,
        'best_dev_rmse': best_dev_rmse,
        'epochs_without_improvement': epochs_without_improvement,
    }

    torch.save(state, RUN_DIR / f'checkpoint_epoch_{epoch}.pt')
    torch.save(state, latest_checkpoint)
    pd.DataFrame(history).to_csv(RUN_DIR / 'history.csv', index=False)

    print(json.dumps(epoch_row, indent=2))
    print(f"Saved epoch checkpoint to: {(RUN_DIR / f'checkpoint_epoch_{epoch}.pt').resolve()}")

    if dev_metrics['rmse'] < best_dev_rmse:
        best_dev_rmse = dev_metrics['rmse']
        epochs_without_improvement = 0
        state['best_dev_rmse'] = best_dev_rmse
        torch.save(state, best_checkpoint)
        print(f'Saved best checkpoint to: {best_checkpoint.resolve()}')
    else:
        epochs_without_improvement += 1
        if epochs_without_improvement >= EARLY_STOPPING_PATIENCE:
            print('Early stopping triggered.')
            break


Epoch 1/5: 100%|██████████| 2841/2841 [41:23<00:00,  1.14it/s, loss=0.455]


{
  "epoch": 1,
  "train_loss": 0.4546379517590603,
  "train_rmse": 5.445472572633769,
  "train_mae": 4.450985431671143,
  "train_pearson_r": -0.0705737948873201,
  "dev_rmse": 6.622669889834957,
  "dev_mae": 5.489506721496582,
  "dev_pearson_r": 0.018641725209360673
}
Saved epoch checkpoint to: /Users/roshan/Documents/SUTD/Term 8/DL/Project/sutd_50.039_Deep_Learning/processed/wav2vec_finetune/wav2vec2_ft_seg8_hop4_u4_lr5e-06/checkpoint_epoch_1.pt
Saved best checkpoint to: /Users/roshan/Documents/SUTD/Term 8/DL/Project/sutd_50.039_Deep_Learning/processed/wav2vec_finetune/wav2vec2_ft_seg8_hop4_u4_lr5e-06/best_model.pt


Epoch 2/5: 100%|██████████| 2841/2841 [42:14<00:00,  1.12it/s, loss=0.282]


{
  "epoch": 2,
  "train_loss": 0.2816704031454045,
  "train_rmse": 5.444369306932489,
  "train_mae": 4.497331142425537,
  "train_pearson_r": 0.21883570439230357,
  "dev_rmse": 6.535539158686647,
  "dev_mae": 5.50484561920166,
  "dev_pearson_r": 0.40475607050722573
}
Saved epoch checkpoint to: /Users/roshan/Documents/SUTD/Term 8/DL/Project/sutd_50.039_Deep_Learning/processed/wav2vec_finetune/wav2vec2_ft_seg8_hop4_u4_lr5e-06/checkpoint_epoch_2.pt
Saved best checkpoint to: /Users/roshan/Documents/SUTD/Term 8/DL/Project/sutd_50.039_Deep_Learning/processed/wav2vec_finetune/wav2vec2_ft_seg8_hop4_u4_lr5e-06/best_model.pt


Epoch 3/5: 100%|██████████| 2841/2841 [42:11<00:00,  1.12it/s, loss=0.28]  


{
  "epoch": 3,
  "train_loss": 0.2797060712120656,
  "train_rmse": 5.429433341904528,
  "train_mae": 4.476826190948486,
  "train_pearson_r": 0.31279858489334106,
  "dev_rmse": 6.545790844283035,
  "dev_mae": 5.4896931648254395,
  "dev_pearson_r": 0.3431045405596218
}
Saved epoch checkpoint to: /Users/roshan/Documents/SUTD/Term 8/DL/Project/sutd_50.039_Deep_Learning/processed/wav2vec_finetune/wav2vec2_ft_seg8_hop4_u4_lr5e-06/checkpoint_epoch_3.pt


Epoch 4/5: 100%|██████████| 2841/2841 [42:14<00:00,  1.12it/s, loss=0.279] 


{
  "epoch": 4,
  "train_loss": 0.2794640054468155,
  "train_rmse": 5.336850715114666,
  "train_mae": 4.365111827850342,
  "train_pearson_r": 0.3571214953770141,
  "dev_rmse": 6.453173177691677,
  "dev_mae": 5.3481831550598145,
  "dev_pearson_r": 0.39103208195206424
}
Saved epoch checkpoint to: /Users/roshan/Documents/SUTD/Term 8/DL/Project/sutd_50.039_Deep_Learning/processed/wav2vec_finetune/wav2vec2_ft_seg8_hop4_u4_lr5e-06/checkpoint_epoch_4.pt
Saved best checkpoint to: /Users/roshan/Documents/SUTD/Term 8/DL/Project/sutd_50.039_Deep_Learning/processed/wav2vec_finetune/wav2vec2_ft_seg8_hop4_u4_lr5e-06/best_model.pt


Epoch 5/5: 100%|██████████| 2841/2841 [42:26<00:00,  1.12it/s, loss=0.275] 


{
  "epoch": 5,
  "train_loss": 0.27515124645318856,
  "train_rmse": 5.306987101083293,
  "train_mae": 4.335944175720215,
  "train_pearson_r": 0.3652139898810542,
  "dev_rmse": 6.4014124027054935,
  "dev_mae": 5.307743549346924,
  "dev_pearson_r": 0.3807208656368343
}
Saved epoch checkpoint to: /Users/roshan/Documents/SUTD/Term 8/DL/Project/sutd_50.039_Deep_Learning/processed/wav2vec_finetune/wav2vec2_ft_seg8_hop4_u4_lr5e-06/checkpoint_epoch_5.pt
Saved best checkpoint to: /Users/roshan/Documents/SUTD/Term 8/DL/Project/sutd_50.039_Deep_Learning/processed/wav2vec_finetune/wav2vec2_ft_seg8_hop4_u4_lr5e-06/best_model.pt


In [47]:
history_df = pd.DataFrame(history)
display(history_df)

checkpoint = torch.load(best_checkpoint, map_location=DEVICE)
model.load_state_dict(checkpoint['model_state_dict'])
model.to(DEVICE)
model.eval()

final_metrics = []
for split in SPLITS:
    predictions, metrics = evaluate_participant_level(model, eval_loaders[split])
    predictions.to_csv(RUN_DIR / f'{split}_participant_predictions.csv', index=False)
    final_metrics.append({'split': split, **metrics})
    print(
        f"{split:5s} | RMSE={metrics['rmse']:.4f} | MAE={metrics['mae']:.4f} | r={metrics['pearson_r']:.4f}"
    )

final_metrics_df = pd.DataFrame(final_metrics)
history_df.to_csv(RUN_DIR / 'history.csv', index=False)
final_metrics_df.to_csv(RUN_DIR / 'participant_metrics.csv', index=False)
participant_df.to_csv(RUN_DIR / 'participant_index.csv', index=False)

display(final_metrics_df)
print(f'Saved run artifacts to: {RUN_DIR.resolve()}')


,epoch,train_loss,train_rmse,train_mae,train_pearson_r,dev_rmse,dev_mae,dev_pearson_r
0,1,0.454638,5.445473,4.450985,-0.070574,6.622670,5.489507,0.018642
1,2,0.281670,5.444369,4.497331,0.218836,6.535539,5.504846,0.404756
2,3,0.279706,5.429433,4.476826,0.312799,6.545791,5.489693,0.343105
3,4,0.279464,5.336851,4.365112,0.357121,6.453173,5.348183,0.391032
4,5,0.275151,5.306987,4.335944,0.365214,6.401412,5.307744,0.380721


KeyboardInterrupt: 

## Notes

- This notebook fine-tunes against `phq_score` as a regression target.
- Segment labels are inherited from the participant label, then averaged back to the participant level for evaluation.
- The weighted loss keeps long recordings from dominating training just because they produce more segments.
- The feature extractor remains frozen; the feature projection, encoder layer norm, last transformer blocks, projector, and regression head are trainable.
- If memory is tight on MPS, reduce `TRAIN_BATCH_SIZE` first before changing the rest of the pipeline.
